
## 🎯 과제 3: 페르소나를 JSON 파일로 분리 (실무 빈도 ★★★★☆)

실무에서는 페르소나가 자주 추가·수정됩니다. 코드 안에 박혀 있으면 매번 코드를 고쳐야 해요. **외부 파일로 분리하면 비개발자도 페르소나를 편집할 수 있습니다.**

**요구사항**
- `personas.json` 파일에 페르소나 정의 저장
- 챗봇 시작 시 JSON을 읽어와 사용

**힌트**
```python
import json
with open("personas.json", encoding="utf-8") as f:
    PERSONAS = json.load(f)
```


In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


# 패키지 import
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")


✅ 로컬 Jupyter
✅ API 키 OK
✅ import 완료


In [10]:
import json
with open("personas.json", encoding="utf-8") as f:
    PERSONAS = json.load(f)

In [11]:
def ask(persona_name: str, question: str) -> str:
    config = PERSONAS[persona_name]
    persona_chain = prompt | llm.bind(
        temperature=config["temperature"]
    )
    result = persona_chain.invoke({
        "role": config["system"],
        "question": question,
    })
    return result.content

# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

In [12]:
llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"),
    ("human", "{question}"),
])

In [16]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 1. **DataDive**: 데이터의 깊이를 탐험하며 분석 기술을 배울 수 있는 앱입니다.

2. **InsightQuest**: 통찰력을 얻기 위한 여정을 통해 데이터 분석의 기초를 익히는 플랫폼입니다.

3. **StatStart**: 통계의 기초부터 시작하여 데이터 분석의 세계로 나아가는 첫걸음 앱입니다.

4. **AnalyzeIt**: 데이터를 분석하는 방법을 직관적으로 배우고 실습할 수 있는 앱입니다.

5. **VizLearn**: 데이터 시각화를 통해 분석 결과를 쉽게 이해하고 배울 수 있는 학습 도구입니다.

✅ 페르소나가 '디자이너'(으)로 변경되었습니다.
[디자이너] 온라인 쇼핑몰에서 장바구니에 상품을 담고도 구매하지 않는 사용자를 줄이기 위해 다음 세 가지를 우선순위에 따라 제안합니다:

1. **장바구니 리마인더 알림 개선하기**
   - **이유**: 사용자가 장바구니에 담긴 상품을 잊어버리는 경우가 많습니다. 이메일이나 푸시 알림을 통해 "장바구니에 상품이 남아있어요!"라는 친절한 알림을 주면, 잊혀진 상품을 다시 떠올리게 할 수 있습니다. 이런 리마인더는 사용자에게 귀찮은 푸시가 아닌, 유용한 정보를 제공하는 느낌을 줄 수 있습니다.

2. **구매 유도 프로모션 제공하기**
   - **이유**: 장바구니에 담긴 상품에 대한 할인이나 무료 배송 같은 프로모션을 제공하면, 사용자의 구매를 유도할 수 있습니다. 이러한 인센티브는 사용자가 결제 버튼을 누르게끔 마음을 움직일 수 있는 달콤한 유혹과 같습니다. 가격에 민감한 소비자에게는 특히 효과적입니다.

3. **간편한 결제 프로세스 설계하기**
   - **이유**: 결제 과정이 복잡하거나 시간이 오래 걸리면 사용자는 쉽게 포기할 수 있습니다. 체크아웃 과정에서 단계 수를 최소화하고, 다양한 결제 옵션을 제공하는 것은 마치 빠르고 부드러운 길을 만드는 것과 같습니다. 사용자